In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import os
from tqdm import tqdm
from random import random as rnd_basic_fnc
from time import sleep
import re

# 1. Получение данных
* Файлов статей и журналов
* информации о ключивых слов
* информации о физических классификаторов (например PACS)

## 1.1 Получение информации из сайта журнала [УФН](ufn.ru)

Построение ссылок год том статья:
> Ссылка на общую страницу тома 12 года 2017, где содежится инфориация о количестве статей рубрике к которой относится статья и о названии статьи
>
> 'https://ufn.ru/ru/articles/2017/12/'

> Ссылка на страницу на сайте первой статьи в номере 12 2017 года (нумирация статей в выпуске идет латинскими буквами), где содежится инфориация о названии статьи авторах аннотации, ключевых словах и номерах PACS
>
> 'https://ufn.ru/ru/articles/2017/12/a/'

> Ссылка на pdf файл первой статьи 11 тома 2017 года
>
> 'https://ufn.ru/ufn17/ufn17_11/Russian/r1711a.pdf'

In [3]:
PATH_1 = 'https://ufn.ru/ru/articles/2017/12/'
PATH_2 = 'https://ufn.ru/ru/articles/2017/12/a/'
PATH_3 = 'https://ufn.ru/ufn17/ufn17_11/Russian/r1711a.pdf' # https://ufn.ru/ufn46/ufn46_11/Russian

### 1.1.1 **Загрузка pdf документов статей**

In [4]:
def download_pdf(url, save_path):
    try:
        # Отправляем запрос на скачивание
        response = requests.get(url, stream=True)
        # Проверяем, успешен ли запрос
        response.raise_for_status()

        # Записываем контент в файл
        with open(save_path, 'wb') as file:
            file.write(response.content)
        print(f"Файл успешно скачан: {save_path}")

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при скачивании: {e}")

In [5]:
# URL вашего PDF-файла
url = PATH_3
# Локальный путь для сохранения
save_path = "document.pdf"

download_pdf(url, save_path)



Файл успешно скачан: document.pdf


### 1.1.2 **Получение данных со страниц**

In [6]:


# 1. Отправляем запрос к сайту
url = PATH_1
response = requests.get(url)

# Проверяем, что запрос прошел успешно
if response.status_code == 200:
    # 2. Передаем HTML в BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # 3. Ищем данные
    # Пример: получить заголовок страницы
    title = soup.title.text
    print(f"Заголовок: {title}")

    # Пример: найти все теги <h2>
    headers = soup.find_all('h2')
    for header in headers:
        print(header.text.strip())


Заголовок: Выпуск 12, 2017


In [7]:
articles = soup.find_all('p', class_='articles')

# Вариант Б: Использование CSS-селектора
# articles = soup.select('p.articles')


articles

cnt = 0
for article in articles:
    cnt += 1
    print(article)
    # print(article.text.strip())
    #print(cnt)


<p class="articles">
А.А. Балакин, Г.М. Фрайман «<a href="/ru/articles/2017/12/a/">Э­лектрон-ионные столкновения в сильных электромагнитных полях</a>» <i>УФН</i> <b>187</b> 1289–1328 (2017)
<br/><br/>
Б.М. Смирнов «<a href="/ru/articles/2017/12/b/">Металлические наноструктуры: от кластеров к нанокатализу и сенсорам</a>» <i>УФН</i> <b>187</b> 1329–1364 (2017)
</p>
<p class="articles">
С.М. Стишов, А.Е. Петрова «<a href="/ru/articles/2017/12/c/">Геликоидальный зонный магнетик MnSi: магнитный фазовый переход</a>» <i>УФН</i> <b>187</b> 1365–1374 (2017)
</p>
<p class="articles">
А.Б. Александров, М.С. Владимиров, В.И. Галкин, Л.А. Гончарова, В.М. Грачёв, С.Г. Васина, Н.С. Коновалова, А.А. Маловичко, А.К. Манагадзе, Н.М. Окатьева, Н.Г. Полухина, Т.М. Роганова, Н.И. Старков, В.Э. Тюков, М.М. Чернявский, Т.В. Щедрина «<a href="/ru/articles/2017/12/d/">Метод мюонной радиографии для фундаментальных и прикладных исследований</a>» <i>УФН</i> <b>187</b> 1375–1392 (2017)
</p>
<p class="articles">
В.

In [8]:
articles = soup.find_all('p', class_='articles')
rubrics = soup.find_all('p', class_='rubric')

# Вариант Б: Использование CSS-селектора
# articles = soup.select('p.articles')


cnt = 0
for iarticle,irubric in zip(articles, rubrics):
    cnt += 1
    titles_with_links = iarticle.find_all('a')
    print(irubric.text.strip())
    for ititle in titles_with_links:
      print(ititle.text.strip())
      print(ititle.get('href'))

Обзоры актуальных проблем
Э­лектрон-ионные столкновения в сильных электромагнитных полях
/ru/articles/2017/12/a/
Металлические наноструктуры: от кластеров к нанокатализу и сенсорам
/ru/articles/2017/12/b/
Физика наших дней
Геликоидальный зонный магнетик MnSi: магнитный фазовый переход
/ru/articles/2017/12/c/
Приборы и методы исследований
Метод мюонной радиографии для фундаментальных и прикладных исследований
/ru/articles/2017/12/d/
Методические заметки
Можно ли измерить электромагнитное излучение внезапно стартующего заряда?
/ru/articles/2017/12/e/
Персоналии
Юрий Алексеевич Трутнев (к 90-летию со дня рождения)
/ru/articles/2017/12/f/
Сергей Михайлович Стишов (к 80-летию со дня рождения)
/ru/articles/2017/12/g/
Памяти Льва Николаевича Липатова
/ru/articles/2017/12/h/
Библиография
Новые книги по физике и смежным наукам
/ru/articles/2017/12/i/
Новости физики в интернете
Новости физики в сети Internet (по материалам электронных препринтов)
/ru/articles/2017/12/j/
Библиография
Годовой указ

In [9]:
title_list = []
href_list = []
rubric_list = []


articles = soup.find_all('p', class_='articles')
rubrics = soup.find_all('p', class_='rubric')

cnt = 0
for iarticle,irubric in zip(articles, rubrics):
    cnt += 1
    titles_with_links = iarticle.find_all('a')
    irubric_name = irubric.text.strip()

    for ititle in titles_with_links:
      title_list.append(ititle.text.strip().replace('\xa0', "").replace('\xad', ""))
      href_list.append(ititle.get('href'))
      rubric_list.append(irubric_name)

### 1.1.3 Обработка индивидуальных веб-страниц УФН-статей
> получение PACS

In [10]:
url = 'https://ufn.ru/ru/articles/2017/12/d/'
response_art = requests.get(url)

if response.status_code == 200:
    # 2. Передаем HTML в BeautifulSoup
    soup_art = BeautifulSoup(response_art.text, 'html.parser')

    # 3. Ищем данные
    # Пример: получить заголовок страницы
    title = soup_art.title.text
    print(f"Заголовок: {title}")

    # Пример: найти все теги <h2>
    headers = soup_art.find_all('h2')
    for header in headers:
        print(header.text.strip())

Заголовок: Метод мюонной радиографии для фундаментальных и прикладных исследований


In [11]:
import re
links = soup_art.find_all('a', href=re.compile(r"/pacs/"))
for ilink in links:
  if re.search(r'\d', ilink.text):  # Вернет True, если есть цифра
    print(ilink.text,ilink['title'].lower() )

07.05.Fb design of experiments
07.05.Hd data acquisition: hardware and software
07.05.Kf data analysis: algorithms and implementation; data management


> doi

In [12]:
import re
links = soup_art.find_all('a', href=re.compile(r"10.3367/UFNr"))
for ilink in links:
  print(ilink.text)

10.3367/UFNr.2017.07.038188


In [13]:
import re
links = soup_art.find_all('a', href=re.compile(r"doi"))
for ilink in links:
  print(ilink.text)

10.3367/UFNe.2017.07.038188
10.3367/UFNr.2017.07.038188
10.3367/UFNe.2017.07.038188


In [15]:
[ival for ival in 'Data acquisition: hardware and software'.split(':')]

['Data acquisition', ' hardware and software']

### 1.1.4 Генерация страниц для обхода

In [16]:
path_to_vol = lambda year, vol : f'https://ufn.ru/ru/articles/{year}/{vol}/'
path_to_art = lambda year, vol, lett : f'https://ufn.ru/ru/articles/{year}/{vol}/{lett}/'
path_to_pdf = lambda year, vol, lett : f'https://ufn.ru/ufn{str(year)[2:]}/ufn{str(year)[2:]}_{vol}/Russian/r{str(year)[2:]}{vol}{lett}.pdf'

In [17]:
path_to_pdf(2012,3,'b')

'https://ufn.ru/ufn12/ufn12_3/Russian/r123b.pdf'

Программа чтообы тормозить

In [18]:
for inumber in tqdm([1,2,3,4,5]):
  wait_vals = 10 + rnd_basic_fnc()*10
  print(f'{wait_vals:.2f}')
  sleep(wait_vals)

  0%|          | 0/5 [00:00<?, ?it/s]

14.77


 20%|██        | 1/5 [00:14<00:59, 14.77s/it]

15.35


 40%|████      | 2/5 [00:30<00:45, 15.11s/it]

...

 60%|██████    | 3/5 [00:48<00:32, 16.48s/it]

11.43


 80%|████████  | 4/5 [00:59<00:14, 14.49s/it]

11.94


100%|██████████| 5/5 [01:11<00:00, 14.32s/it]


In [19]:
wait_sec_func = lambda sec_1, sec_2 : sleep(sec_1 + rnd_basic_fnc()*(sec_2-sec_1))

wait_sec_func(1,2)

## 1.2 Скачивание данных

In [23]:
# from google.colab import drive
# drive.mount('/content/drive')

In [21]:
PATH_DISK = '/content/drive/MyDrive/1.ru_ph_BERT/'
FOLDER_NAME = 'data/ufn/'

# os.makedirs('/content/drive/MyDrive/1.ru_ph_BERT/data/ufn/')

Тестирование

In [22]:
path_to_vol = lambda year, vol : f'https://ufn.ru/ru/articles/{year}/{vol}/'
path_to_art = lambda year, vol, lett : f'https://ufn.ru/ru/articles/{year}/{vol}/{lett}/'
path_to_pdf = lambda year, vol, lett : f'https://ufn.ru/ufn{str(year)[2:]}/ufn{str(year)[2:]}_{vol}/Russian/r{str(year)[2:]}{vol}{lett}.pdf'
path_to_disk = lambda year, vol, lett : f'/content/drive/MyDrive/1.ru_ph_BERT/data/ufn/r{str(year)[2:]}{vol}{lett}.pdf'
name_pdf = lambda year, vol, lett : f'r{str(year)[2:]}{vol}{lett}.pdf'
def download_pdf(url, save_path):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(save_path, 'wb') as file:
            file.write(response.content)
        status_flag = True
    except requests.exceptions.RequestException as e:
        status_flag = False

    return status_flag

info_rubric_list = []
info_title_list = []

info_vol_list = []
info_year_list = []

info_pacs_list = []
info_pacs_text_list = []
info_doi_list = []
info_href_list = []


info_pdf_name_list = []
info_pdf_dwnload_status = []

YEAR = 2017
VOLUME = 12


# processing volume web-page
url_volume = path_to_vol(YEAR,VOLUME) # 'https://ufn.ru/ru/articles/2017/12/'
response = requests.get(url_volume)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')


    articles = soup.find_all('p', class_='articles')
    rubrics = soup.find_all('p', class_='rubric')

    for iarticle,irubric in zip(articles, rubrics):
        titles_with_links = iarticle.find_all('a')
        iarts_rubric = irubric.text.strip()
        for ititle in titles_with_links:
          ititle_of_art = ititle.text.strip()
          # print(ititle.get('href'))

          # processing article web-page
          ilink_article_webpage = 'https://ufn.ru' + ititle.get('href')
          art_letter = ilink_article_webpage.split('/')[-2]

          ilink_article_pdf = path_to_pdf( YEAR,
                                           VOLUME,
                                           art_letter)

          idisk_pdf = path_to_disk( YEAR,
                                    VOLUME,
                                    art_letter)

          iname_pdf = name_pdf( YEAR,
                                VOLUME,
                                art_letter)

          response_art = requests.get(ilink_article_webpage)
          local_pacs = ''
          local_pacs_text = ''

          if response_art.status_code == 200:
              soup_art = BeautifulSoup(response_art.text, 'html.parser')
              links = soup_art.find_all('a', href=re.compile(r"/pacs/"))
              links_doi = soup_art.find_all('a', href=re.compile(r"doi"))

              local_pacs = []
              local_pacs_text = []
              for ilink in links:
                if re.search(r'\d', ilink.text):  # True if digital in string
                  local_pacs.append(ilink.text)
                  local_pacs_text.append(ilink['title'].lower())
              local_pacs = '||'.join(local_pacs)
              local_pacs_text = '||'.join(local_pacs_text)

              local_doi = []
              for ilink in links_doi:
                local_doi.append(ilink.text)
              iart_doi = '||'.join(list(set(local_doi)))
              info_pacs_list.append(local_pacs)
              info_pacs_text_list.append(local_pacs_text)
              info_doi_list.append(iart_doi)

          else:
              info_pacs_list.append(False)
              info_pacs_text_list.append(False)
              info_doi_list.append(False)

          info_rubric_list.append(iarts_rubric)
          info_title_list.append(ititle_of_art)
          info_vol_list.append(VOLUME)
          info_year_list.append(YEAR)
          info_href_list.append(ilink_article_webpage)

          dowload_status = download_pdf(ilink_article_pdf, idisk_pdf)

          info_pdf_name_list.append(iname_pdf)
          info_pdf_dwnload_status.append(dowload_status)

        break

          # download article web-page

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/1.ru_ph_BERT/data/ufn/r1712a.pdf'

In [ ]:
# for ival in [info_rubric_list,
# info_title_list,
# info_vol_list,
# info_year_list,
# info_pacs_list,
# info_pacs_text_list,
# info_doi_list,
# info_href_list,
# info_pdf_name_list,
# info_pdf_dwnload_status]:
#   print(ival)

['Обзоры актуальных проблем', 'Обзоры актуальных проблем']
['Э\xadлектрон-ионные столкновения в сильных электромагнитных полях', 'Металлические наноструктуры: от кластеров к нанокатализу и сенсорам']
[12, 12]
[2017, 2017]
['52.20.Dq||52.20.Fs||52.25.Dg||52.50.Sw', '61.43.Hv||61.46.−w||72.15.−v||73.63.−b']
['particle orbits||electron collisions||plasma kinetic equations||plasma heating by microwaves; ecr, lh, collisional heating', 'fractals; macroscopic aggregates (including diffusion-limited aggregates)||structure of nanoscale materials||electronic conduction in metals and alloys||electronic transport in nanoscale materials and structures']
['10.3367/UFNe.2017.02.038075||10.3367/UFNr.2017.02.038075', '10.3367/UFNe.2017.02.038073||10.3367/UFNr.2017.02.038073']
['https://ufn.ru/ru/articles/2017/12/a/', 'https://ufn.ru/ru/articles/2017/12/b/']
['r1712a.pdf', 'r1712b.pdf']
[True, True]


Основной проход

In [ ]:
path_to_vol = lambda year, vol : f'https://ufn.ru/ru/articles/{year}/{vol}/'
path_to_art = lambda year, vol, lett : f'https://ufn.ru/ru/articles/{year}/{vol}/{lett}/'
path_to_pdf = lambda year, vol, lett : f'https://ufn.ru/ufn{str(year)[2:]}/ufn{str(year)[2:]}_{vol}/Russian/r{str(year)[2:]}{vol}{lett}.pdf'
path_to_disk = lambda year, vol, lett : f'/content/drive/MyDrive/1.ru_ph_BERT/data/ufn/r{str(year)[2:]}{vol}{lett}.pdf'
name_pdf = lambda year, vol, lett : f'r{str(year)[2:]}{vol}{lett}.pdf'

wait_sec_func = lambda sec_1, sec_2 : sleep(sec_1 + rnd_basic_fnc()*(sec_2-sec_1))



def download_pdf(url, save_path):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(save_path, 'wb') as file:
            file.write(response.content)
        status_flag = True
    except requests.exceptions.RequestException as e:
        status_flag = False

    return status_flag

info_rubric_list = []
info_title_list = []

info_vol_list = []
info_year_list = []

info_pacs_list = []
info_pacs_text_list = []
info_doi_list = []
info_href_list = []


info_pdf_name_list = []
info_pdf_dwnload_status = []





pbar = tqdm(range(2025,2026))

for iYEAR in pbar:
    # year waiting times  10-20 sec
    # wait_sec_func(10,20)

    for iVOL in range(1,13):
      # volume waiting times  5-10 sec
      # wait_sec_func(5,10)

      YEAR = int(iYEAR)
      VOLUME = int(iVOL)

      # processing volume web-page
      url_volume = path_to_vol(YEAR,VOLUME) # 'https://ufn.ru/ru/articles/2017/12/'
      response = requests.get(url_volume)

      if response.status_code == 200:
          soup = BeautifulSoup(response.text, 'html.parser')


          articles = soup.find_all('p', class_='articles')
          rubrics = soup.find_all('p', class_='rubric')

          for iarticle,irubric in zip(articles, rubrics):
              titles_with_links = iarticle.find_all('a')
              iarts_rubric = irubric.text.strip()
              for ititle in titles_with_links:
                ititle_of_art = ititle.text.strip()
                # print(ititle.get('href'))

                # processing article web-page
                ilink_article_webpage = 'https://ufn.ru' + ititle.get('href')
                art_letter = ilink_article_webpage.split('/')[-2]

                ilink_article_pdf = path_to_pdf( YEAR,
                                                VOLUME,
                                                art_letter)

                idisk_pdf = path_to_disk( YEAR,
                                          VOLUME,
                                          art_letter)

                iname_pdf = name_pdf( YEAR,
                                      VOLUME,
                                      art_letter)

                response_art = requests.get(ilink_article_webpage)
                local_pacs = ''
                local_pacs_text = ''

                if response_art.status_code == 200:
                    soup_art = BeautifulSoup(response_art.text, 'html.parser')
                    links = soup_art.find_all('a', href=re.compile(r"/pacs/"))
                    links_doi = soup_art.find_all('a', href=re.compile(r"doi"))

                    local_pacs = []
                    local_pacs_text = []
                    for ilink in links:
                      if re.search(r'\d', ilink.text):  # True if digital in string
                        local_pacs.append(ilink.text)
                        local_pacs_text.append(ilink['title'].lower())
                    local_pacs = '||'.join(local_pacs)
                    local_pacs_text = '||'.join(local_pacs_text)

                    local_doi = []
                    for ilink in links_doi:
                      local_doi.append(ilink.text)
                    iart_doi = '||'.join(list(set(local_doi)))
                    info_pacs_list.append(local_pacs)
                    info_pacs_text_list.append(local_pacs_text)
                    info_doi_list.append(iart_doi)

                else:
                    info_pacs_list.append(False)
                    info_pacs_text_list.append(False)
                    info_doi_list.append(False)

                info_rubric_list.append(iarts_rubric)
                info_title_list.append(ititle_of_art)
                info_vol_list.append(VOLUME)
                info_year_list.append(YEAR)
                info_href_list.append(ilink_article_webpage)

                dowload_status = download_pdf(ilink_article_pdf, idisk_pdf)

                info_pdf_name_list.append(iname_pdf)
                info_pdf_dwnload_status.append(dowload_status)
                pbar.set_description(f'{YEAR}_{VOLUME:02d}_{art_letter}')
                # article waiting times  1-2 sec
                # wait_sec_func(1,2)


2024_12_n: 100%|██████████| 27/27 [2:03:19<00:00, 274.05s/it]


In [ ]:
info_df = pd.DataFrame({
    'title':info_title_list,
    'rubr':info_rubric_list,
    'vol':info_vol_list,
    'year':info_year_list,
    'pacs':info_pacs_list,
    'pacs_txt':info_pacs_text_list,
    'doi':info_doi_list,
    'link':info_href_list,
    'pdf':info_pdf_name_list,
    'dwl':info_pdf_dwnload_status,
})

In [ ]:
info_df['dwl'].mean()

np.float64(1.0)

In [ ]:
path_to_vol = lambda year, vol : f'https://ufn.ru/ru/articles/{year}/{vol}/'
path_to_art = lambda year, vol, lett : f'https://ufn.ru/ru/articles/{year}/{vol}/{lett}/'
path_to_pdf = lambda year, vol, lett : f'https://ufn.ru/ufn{str(year)[2:]}/ufn{str(year)[2:]}_{vol}/Russian/r{str(year)[2:]}{vol}{lett}.pdf'
path_to_disk = lambda year, vol, lett : f'/content/drive/MyDrive/1.ru_ph_BERT/data/ufn/r{str(year)[2:]}{vol}{lett}.pdf'
name_pdf = lambda year, vol, lett : f'r{str(year)[2:]}{vol}{lett}.pdf'

wait_sec_func = lambda sec_1, sec_2 : sleep(sec_1 + rnd_basic_fnc()*(sec_2-sec_1))



def download_pdf(url, save_path):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(save_path, 'wb') as file:
            file.write(response.content)
        status_flag = True
    except requests.exceptions.RequestException as e:
        status_flag = False

    return status_flag

info_rubric_list = []
info_title_list = []

info_vol_list = []
info_year_list = []

info_pacs_list = []
info_pacs_text_list = []
info_doi_list = []
info_href_list = []


info_pdf_name_list = []
info_pdf_dwnload_status = []





pbar = tqdm(range(1960,2026))

for iYEAR in pbar:
    # year waiting times  10-20 sec
    # wait_sec_func(10,20)

    for iVOL in range(1,13):
      # volume waiting times  5-10 sec
      # wait_sec_func(5,10)

      YEAR = int(iYEAR)
      VOLUME = int(iVOL)

      # processing volume web-page
      url_volume = path_to_vol(YEAR,VOLUME) # 'https://ufn.ru/ru/articles/2017/12/'
      response = requests.get(url_volume)

      if response.status_code == 200:
          soup = BeautifulSoup(response.text, 'html.parser')


          articles = soup.find_all('p', class_='articles')
          rubrics = soup.find_all('p', class_='rubric')

          for iarticle,irubric in zip(articles, rubrics):
              titles_with_links = iarticle.find_all('a')
              iarts_rubric = irubric.text.strip()
              for ititle in titles_with_links:
                ititle_of_art = ititle.text.strip()
                # print(ititle.get('href'))

                # processing article web-page
                ilink_article_webpage = 'https://ufn.ru' + ititle.get('href')
                art_letter = ilink_article_webpage.split('/')[-2]

                ilink_article_pdf = path_to_pdf( YEAR,
                                                VOLUME,
                                                art_letter)

                idisk_pdf = path_to_disk( YEAR,
                                          VOLUME,
                                          art_letter)

                iname_pdf = name_pdf( YEAR,
                                      VOLUME,
                                      art_letter)

                response_art = requests.get(ilink_article_webpage)
                local_pacs = ''
                local_pacs_text = ''

                if response_art.status_code == 200:
                    soup_art = BeautifulSoup(response_art.text, 'html.parser')
                    links = soup_art.find_all('a', href=re.compile(r"/pacs/"))
                    links_doi = soup_art.find_all('a', href=re.compile(r"doi"))

                    local_pacs = []
                    local_pacs_text = []
                    for ilink in links:
                      if re.search(r'\d', ilink.text):  # True if digital in string
                        local_pacs.append(ilink.text)
                        local_pacs_text.append(ilink['title'].lower())
                    local_pacs = '||'.join(local_pacs)
                    local_pacs_text = '||'.join(local_pacs_text)

                    local_doi = []
                    for ilink in links_doi:
                      local_doi.append(ilink.text)
                    iart_doi = '||'.join(list(set(local_doi)))
                    info_pacs_list.append(local_pacs)
                    info_pacs_text_list.append(local_pacs_text)
                    info_doi_list.append(iart_doi)

                else:
                    info_pacs_list.append(False)
                    info_pacs_text_list.append(False)
                    info_doi_list.append(False)

                info_rubric_list.append(iarts_rubric)
                info_title_list.append(ititle_of_art)
                info_vol_list.append(VOLUME)
                info_year_list.append(YEAR)
                info_href_list.append(ilink_article_webpage)
                info_pdf_name_list.append(iname_pdf)
                pbar.set_description(f'{YEAR}_{VOLUME:02d}_{art_letter}')
                # article waiting times  1-2 sec
                # wait_sec_func(1,2)
info_df = pd.DataFrame({
    'title':info_title_list,
    'rubr':info_rubric_list,
    'vol':info_vol_list,
    'year':info_year_list,
    'pacs':info_pacs_list,
    'pacs_txt':info_pacs_text_list,
    'doi':info_doi_list,
    'link':info_href_list,
})
info_df.to_csv('/content/drive/MyDrive/1.ru_ph_BERT/data/info_ufn.csv')

2025_12_j: 100%|██████████| 66/66 [1:44:29<00:00, 94.99s/it]


In [ ]:
info_df.info()